# 📚 LLaMA-based RAG System — Multi-PDF Question Answering

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline using:
- 🦙 **LLaMA 3 8B Instruct** (4-bit quantized) as the LLM
- 🗄️ **FAISS** for fast vector similarity search
- 🔤 **sentence-transformers** for embeddings
- 📄 **pypdf** for PDF text extraction
- 🎛️ **Gradio** for an interactive chat UI

> ⚠️ **Before running:** Set Runtime to **GPU (T4)** via `Runtime → Change runtime type`

In [ ]:
# ── Step 1: Install dependencies
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    accelerate \
    bitsandbytes \
    gradio \
    huggingface_hub

print('✅ All packages installed.')

## Step 2 — HuggingFace Login

1. Accept the LLaMA 3 license: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
2. Create an access token: https://huggingface.co/settings/tokens
3. Add it to **Colab Secrets** (🔑 left sidebar) as `HF_TOKEN`

In [ ]:
# ── Step 2: Login to HuggingFace
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('✅ Logged in via Colab Secret.')
except Exception:
    login()  # interactive prompt

In [ ]:
# ── Step 3: Imports
!pip install pypdf langchain langchain-text-splitters langchain-community langchain-huggingface transformers
import os, torch, warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from typing import List

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    pipeline, BitsAndBytesConfig,
)
from langchain_community.llms import HuggingFacePipeline

print(f'✅ Imports OK | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import BitsAndBytesConfig

In [ ]:
# ── Step 4: Load LLaMA 3 8B Instruct with 4-bit quantization
# First run downloads ~4.5 GB — allow 5-10 minutes

MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

hf_pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=hf_pipe)
print('✅ LLaMA 3 loaded successfully!')

In [ ]:
# ── Step 5: Load embedding model
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print(f'✅ Embeddings ready: {EMBED_MODEL}')

In [ ]:
# ── Step 6: LlamaRAG class

class LlamaRAG:
    PROMPT_TEMPLATE = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        'You are a precise document assistant. Answer the user ONLY using '
        'the provided context. If the answer is not found, say exactly: '
        '"I could not find this information in the uploaded documents." '
        'Never use outside knowledge. Cite source name and page when helpful.'
        '<|eot_id|>\n'
        '<|start_header_id|>user<|end_header_id|>\n'
        'Context:\n{context}\n\nQuestion: {question}'
        '<|eot_id|>\n'
        '<|start_header_id|>assistant<|end_header_id|>\n'
    )

    def __init__(self, llm, embeddings, chunk_size=512, chunk_overlap=64):
        self.llm = llm
        self.embeddings = embeddings
        self.vectorstore = None
        self.loaded_files = []
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=['\n\n', '\n', '. ', ' ', ''],
        )
        self.prompt = PromptTemplate(
            input_variables=['context', 'question'],
            template=self.PROMPT_TEMPLATE,
        )

    def _parse_pdf(self, path):
        reader = PdfReader(path)
        name = Path(path).name
        docs = []
        for i, page in enumerate(reader.pages):
            text = (page.extract_text() or '').strip()
            if text:
                docs.append(Document(
                    page_content=text,
                    metadata={'source': name, 'page': i + 1},
                ))
        return docs

    def add_pdfs(self, paths):
        chunks_all, added = [], []
        for path in paths:
            if path in self.loaded_files:
                continue
            try:
                raw = self._parse_pdf(path)
                if not raw:
                    print(f'  No text found in {Path(path).name}')
                    continue
                chunks = self.splitter.split_documents(raw)
                chunks_all.extend(chunks)
                self.loaded_files.append(path)
                added.append(Path(path).name)
                print(f'  {Path(path).name}: {len(raw)} pages, {len(chunks)} chunks')
            except Exception as exc:
                print(f'  Error parsing {Path(path).name}: {exc}')
        if not chunks_all:
            return 'No new content indexed.'
        if self.vectorstore is None:
            self.vectorstore = FAISS.from_documents(chunks_all, self.embeddings)
        else:
            self.vectorstore.merge_from(
                FAISS.from_documents(chunks_all, self.embeddings)
            )
        total = self.vectorstore.index.ntotal
        return (f'Indexed {len(added)} file(s): {", ".join(added)}\n'
                f'Vector store: {total} chunks across {len(self.loaded_files)} doc(s).')

    def clear(self):
        self.vectorstore = None
        self.loaded_files = []
        return 'All documents cleared.'

    def status(self):
        if not self.loaded_files:
            return 'No documents loaded.'
        total = self.vectorstore.index.ntotal if self.vectorstore else 0
        lines = [f'{len(self.loaded_files)} doc(s) | {total} chunks']
        for f in self.loaded_files:
            lines.append(f'  - {Path(f).name}')
        return '\n'.join(lines)

    def ask(self, question, k=5, show_sources=True):
        if not question.strip():
            return 'Please enter a question.'
        if self.vectorstore is None:
            return 'No documents indexed yet. Upload PDFs first.'
        docs = self.vectorstore.similarity_search(question, k=k)
        if not docs:
            return 'No relevant content found in the documents.'
        context = '\n\n---\n\n'.join(
            f'[{i}] (Source: {d.metadata.get("source","?")}, '
            f'Page {d.metadata.get("page","?")}\n{d.page_content}'
            for i, d in enumerate(docs, 1)
        )
        answer = self.llm.invoke(
            self.prompt.format(context=context, question=question)
        ).strip()
        if show_sources:
            seen, lines = set(), []
            for d in docs:
                key = (d.metadata.get('source'), d.metadata.get('page'))
                if key not in seen:
                    seen.add(key)
                    lines.append(f'  - {key[0]}, page {key[1]}')
            answer += '\n\nSources:\n' + '\n'.join(lines)
        return answer


rag = LlamaRAG(llm=llm, embeddings=embeddings)
print('RAG system ready!')

In [ ]:
# ── Step 7: Gradio UI
import gradio as gr

def upload_pdfs(files):
    if not files:
        return 'No files selected.', rag.status()
    return rag.add_pdfs([f.name for f in files]), rag.status()

def answer_question(question, history, show_src):
    if not question.strip():
        return history, ""

    history = history or []

    ans = rag.ask(question, k=5, show_sources=show_src)

    history.append(
        {
            "role": "user",
            "content": question
        }
    )

    history.append(
        {
            "role": "assistant",
            "content": ans
        }
    )

    return history, ""

def clear_docs():
    return rag.clear(), rag.status(), []

with gr.Blocks(theme=gr.themes.Soft(), title='LLaMA RAG') as demo:

    gr.Markdown(
        '# LLaMA 3 RAG — Multi-PDF Question Answering\n'
        'Upload PDFs, then ask questions. Answers come **only** from your documents.'
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Document Management')
            file_input    = gr.File(label='Upload PDF(s)', file_types=['.pdf'],
                                    file_count='multiple')
            upload_btn    = gr.Button('Index Documents', variant='primary')
            clear_btn     = gr.Button('Clear All Documents', variant='stop')
            upload_status = gr.Textbox(label='Upload Status', lines=4, interactive=False)
            doc_status    = gr.Textbox(label='Indexed Documents',
                                       value='No documents loaded.',
                                       lines=6, interactive=False)
            show_sources  = gr.Checkbox(label='Show source citations', value=True)

        with gr.Column(scale=2):
            gr.Markdown('### Ask Questions')
            chatbot      = gr.Chatbot(height=460)
            question_box = gr.Textbox(
                label='Your Question',
                placeholder='Ask anything about the uploaded PDFs...',
                lines=2,
            )
            with gr.Row():
                ask_btn  = gr.Button('Ask', variant='primary', scale=3)
                clr_chat = gr.Button('Clear Chat', scale=1)

    upload_btn.click(upload_pdfs, [file_input], [upload_status, doc_status])
    clear_btn.click(clear_docs, [], [upload_status, doc_status, chatbot])
    ask_btn.click(answer_question, [question_box, chatbot, show_sources],
                  [chatbot, question_box])
    question_box.submit(answer_question, [question_box, chatbot, show_sources],
                        [chatbot, question_box])
    clr_chat.click(lambda: [], outputs=[chatbot])

demo.launch(share=True, debug=False, quiet=True)

## Optional — Programmatic Test (no UI)

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # select PDFs
# print(rag.add_pdfs(list(uploaded.keys())))
# print(rag.status())
# print(rag.ask('What is the main topic of this document?'))